# Weak Grid-STRidge and fixed-library stability baselines

These are **baselines**, not the proposed method. They use the weak candidate library but do not use continuous-order best-subset Pareto-DE.

Use this notebook to understand what the baselines are doing on the same dataset/noise/profile controls used by the main tutorial and scripts.


## 0. User controls

In [ ]:
# Robust project-root discovery.
# This avoids failures when a Jupyter kernel is started in a directory that is
# later moved/deleted, in which case Path.cwd() itself can raise FileNotFoundError.
import os
import sys
from pathlib import Path


def find_fpde_project_root() -> Path:
    """Return the repository root containing dataset_configs.py and data/.

    Priority:
    1. FPDE_PROJECT_ROOT environment variable, if set.
    2. Current/PWD directories and their parents, if available.
    3. Common local search locations. This keeps notebooks runnable from
       project root, from notebooks/, and after opening a notebook from an IDE.
    """
    def looks_like_root(path: Path) -> bool:
        return (
            (path / "dataset_configs.py").is_file()
            and (path / "weak_pareto_fde_discovery.py").is_file()
            and (path / "data").is_dir()
        )

    candidates = []
    env_root = os.environ.get("FPDE_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    # os.getcwd() can fail if the kernel's working directory was deleted.
    try:
        candidates.append(Path(os.getcwd()).expanduser())
    except FileNotFoundError:
        pass

    # PWD may still contain a useful absolute path even when os.getcwd() fails.
    pwd = os.environ.get("PWD")
    if pwd:
        candidates.append(Path(pwd).expanduser())

    # Also try the directory containing this notebook if Jupyter exposes it via env.
    for key in ("NOTEBOOK_DIR", "JUPYTER_SERVER_ROOT"):
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    seen = set()
    for cand in candidates:
        try:
            cand = cand.resolve(strict=False)
        except Exception:
            continue
        for path in (cand, cand.parent, *cand.parents):
            if path in seen:
                continue
            seen.add(path)
            if looks_like_root(path):
                return path

    # Conservative bounded search over common project locations.
    search_roots = [
        Path.home() / "Desktop" / "research",
        Path.home() / "Desktop",
        Path.home(),
        Path("/mnt/data"),
    ]
    max_dirs = 5000
    for base in search_roots:
        if not base.exists():
            continue
        visited = 0
        for dirpath, dirnames, filenames in os.walk(base):
            visited += 1
            # Keep the search cheap and avoid hidden/cache directories.
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and d not in {"__pycache__", ".ipynb_checkpoints"}]
            if "dataset_configs.py" in filenames and "weak_pareto_fde_discovery.py" in filenames:
                candidate = Path(dirpath)
                if looks_like_root(candidate):
                    return candidate
            if visited >= max_dirs:
                break

    raise FileNotFoundError(
        "Could not locate the fractional_pareto project root. "
        "Set FPDE_PROJECT_ROOT=/path/to/fractional_pareto_publication_ready_final "
        "or open the notebook from the project root/notebooks directory."
    )


ROOT = find_fpde_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")


import pandas as pd

from dataset_configs import benchmark_spec, config_search_space_fingerprint
from baselines import weak_grid_stridge_baseline, weak_fixed_library_stability_baseline

# EDIT THESE VALUES
dataset_name = "synthetic_time_space_fractional_RD"
noise_percent = 0.5
profile = "notebook"  # use "paper" for final-quality settings
seed = 0

# Candidate-library overrides. In notebook mode we use a small teaching library;
# in paper mode, leave them as None to use the canonical paper library.
cmax_override = 2 if profile == "notebook" else None
p_values_override = (0,) if profile == "notebook" else None

# Runtime overrides. In paper mode, leave these as None.
maxiter_override = 0 if profile == "notebook" else None
popsize_override = 2 if profile == "notebook" else None

weak_test_budget = "smoke" if profile == "notebook" else "paper"
stability_splits = 1 if profile == "notebook" else 5
stability_width_scales = [1.0] if profile == "notebook" else [0.8, 1.0, 1.2]


## 1. Load the same benchmark spec used by the scripts

In [ ]:
spec = benchmark_spec(
    dataset_name,
    data_dir=ROOT / "data",
    profile=profile,
    noise_percent=noise_percent,
    seed=seed,
    maxiter=maxiter_override,
    popsize=popsize_override,
    cmax=cmax_override,
    p_values=p_values_override,
)
data, config = spec["data"], spec["config"]
config.progress = False
config.progress_de = False

print(data.truth)
print("Runtime budget:", {"maxiter": config.maxiter, "popsize": config.popsize})
print("Candidate cmax/p_values:", config.cmax, config.p_values)
print("alpha grid size:", len(config.alpha_grid), "beta grid size:", len(config.beta_grid))
weak_test_budget = "smoke" if profile == "notebook" else "paper"
stability_splits = 1 if profile == "notebook" else 5
stability_width_scales = [1.0] if profile == "notebook" else [0.8, 1.0, 1.2]


## 2. Baseline: weak library + Grid-STRidge

This baseline builds a fixed weak grid library over the configured alpha/beta grids and uses STRidge to select active columns. It tests whether the Pareto-DE selector is useful beyond weak feature construction.


In [ ]:
res = weak_grid_stridge_baseline(
    data,
    config,
    verbose=False,
    max_terms=config.cmax,
    test_budget=weak_test_budget,
)
print(res.equation)
pd.DataFrame([res.to_dict()])

## 3. Baseline/ablation: fixed weak library + stability-selected STRidge

This repeats the fixed-grid weak STRidge selection over weak-test scales/splits and chooses a stable structure. It is meaningful because the candidate library is fixed to a grid; it is not the proposed continuous-order Pareto-DE method.


In [ ]:
stab = weak_fixed_library_stability_baseline(
    data,
    config,
    verbose=False,
    max_terms=config.cmax,
    test_budget=weak_test_budget,
    width_scales=stability_width_scales,
    n_splits=stability_splits,
)
print(stab.equation)
pd.DataFrame([stab.to_dict()])

## 4. How to interpret this baseline

- If `weak_pareto` beats `vanilla_pareto`, the weak library matters.
- If `weak_pareto` beats `weak_grid_stridge`, the best-subset Pareto-DE selector matters.
- If `weak_fixed_stability` is competitive, stability across weak grids is useful, but it is still a fixed-library ablation.
